# Pooled-path test comparison

This notebook does not retrain anything. It copies test metrics that already exist and puts the ones that can be compared in one table.

## What pooled path means

A pooled-path score for horizon *H* is the error over **the whole forecast**, not only the last day.

From each test start date *t*, the model issues *H* daily values: day *t+1*, *t+2*, …, *t+H*. Those days are stacked together and MAE / RMSE / R² (and the rest) are computed on the stack, in original µg/m³. Persistence is scored the same way.

So “7-day pooled path” is 1–7, not “the 7-day-ahead day only”. Same for 14 and 30.

Only start dates that have every lead of that path are kept (*N_origins*). The number of scored days is *N_origins* × *H* (*N_path_points*).

Trees get there by stitching a separate model per lead. CNN-LSTM and the transformer each use one multi-output network for that *H*. The **score** is the same job; the **model** is not.

**1-day** is a separate lead (nothing to pool). CNN-LSTM and the transformer each have a dedicated 1-day run stored in those notebooks. RF and XGBoost use their 1-day specialist on `*_predictions_1d.csv`. LSTM and GRU in `gru_lstm_multi_horizon.ipynb` have no 1-day model.

**Included** (original µg/m³; pooled path 7 / 14 / 30; plus lead 1):

- CNN-LSTM — test `metrics_summary` and the printed 1-day test block in `cnn_lstm.ipynb`
- Transformer — same in `transformer.ipynb`
- LSTM / GRU — `true_multi_horizon_results/multi_horizon_metrics_original_scale.csv` from `gru_lstm_multi_horizon.ipynb` (7 / 14 / 30 only)
- Random Forest — pooled-path CSV plus `rf_predictions_1d.csv`
- XGBoost — pooled-path CSV plus `xgb_predictions_1d.csv`

**Left out:** Direct day-*h* tree tables for h other than 1. Fit/val scores.


In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HERE = Path(".")
CNN_NB = HERE / "cnn_lstm.ipynb"
TRANS_NB = HERE / "transformer.ipynb"
RF_CSV = HERE / "rf_results_path" / "random_forest_pooled_path_results.csv"
XGB_CSV = HERE / "xgb_results_path" / "xgboost_pooled_path_results.csv"
RF_PRED = HERE / "rf_results_path"
XGB_PRED = HERE / "xgb_results_path"
LSTM_GRU_CSV = HERE / "true_multi_horizon_results" / "multi_horizon_metrics_original_scale.csv"
LSTM_GRU_PRED = HERE / "true_multi_horizon_results"


## Read CNN-LSTM and transformer test tables from their notebooks

Those notebooks are not executed. 7 / 14 / 30 come from the stored `metrics_summary`. 1-day comes from the printed `Horizon 1d` test block (CNN-LSTM labels it `Horizon 1d`; the transformer labels it `Horizon 1d (test)`).


In [17]:
def metrics_summary_from_notebook(nb_path):
    """Pull the stored metrics_summary display (test, 7/14/30)."""
    nb = json.loads(Path(nb_path).read_text())
    plains = []
    for cell in nb["cells"]:
        src = "".join(cell.get("source", []))
        if "metrics_summary" not in src or "metrics_7" not in src:
            continue
        for out in cell.get("outputs", []):
            data = out.get("data") or {}
            plain = data.get("text/plain")
            if not plain:
                continue
            if isinstance(plain, list):
                plain = "".join(plain)
            if "MAE" in plain and "7" in plain and "30" in plain:
                plains.append(plain)
    if not plains:
        raise ValueError(f"No metrics_summary output in {nb_path}")
    text = plains[0]
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 7 and parts[0] in {"7", "14", "30"}:
            h, mae, rmse, r2, mape, mse, mbe = parts
            rows.append(
                {
                    "Horizon_Days": int(h),
                    "MAE": float(mae),
                    "RMSE": float(rmse),
                    "R2": float(r2),
                    "MAPE": float(mape),
                    "MSE": float(mse),
                    "MBE": float(mbe),
                }
            )
    df = pd.DataFrame(rows)
    if list(df["Horizon_Days"]) != [7, 14, 30]:
        raise ValueError(f"Unexpected horizons in {nb_path}: {df}")
    return df


def lead1_from_notebook(nb_path):
    """Parse the stored Horizon 1d print (not the 14-day line)."""
    nb = json.loads(Path(nb_path).read_text())
    blob = "\n".join(
        "".join(out.get("text", []))
        for cell in nb["cells"]
        for out in cell.get("outputs", [])
        if out.get("output_type") == "stream"
    )
    key = None
    for candidate in ("Horizon 1d (test)", "Horizon 1d\n"):
        if candidate in blob:
            key = candidate
            break
    if key is None:
        raise ValueError(f"Missing Horizon 1d print in {nb_path}")
    chunk = blob.split(key, 1)[1]
    got = {"Horizon_Days": 1}
    for line in chunk.splitlines()[:8]:
        line = line.strip()
        if line.startswith("MAE:"):
            got["MAE"] = float(line.split()[1])
        elif line.startswith("RMSE:"):
            got["RMSE"] = float(line.split()[1])
        elif line.startswith("R2:") or line.startswith("R"):
            got["R2"] = float(line.split()[1])
        elif line.startswith("MAPE:"):
            got["MAPE"] = float(line.split()[1].replace("%", ""))
        elif line.startswith("MSE:"):
            got["MSE"] = float(line.split()[1])
        elif line.startswith("MBE:"):
            got["MBE"] = float(line.split()[1])
    missing = {"MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"} - set(got)
    if missing:
        raise ValueError(f"Incomplete Horizon 1d metrics in {nb_path}: {missing}")
    return pd.DataFrame([got])


cnn_1 = lead1_from_notebook(CNN_NB)
cnn_1.insert(0, "Model", "CNN-LSTM")
cnn_1["Source"] = "cnn_lstm.ipynb (stored Horizon 1d print)"

transformer_1 = lead1_from_notebook(TRANS_NB)
transformer_1.insert(0, "Model", "Transformer")
transformer_1["Source"] = "transformer.ipynb (stored Horizon 1d (test) print)"

cnn = metrics_summary_from_notebook(CNN_NB)
cnn.insert(0, "Model", "CNN-LSTM")
cnn["Source"] = "cnn_lstm.ipynb (stored test metrics_summary)"
cnn = pd.concat([cnn_1, cnn], ignore_index=True)

transformer = metrics_summary_from_notebook(TRANS_NB)
transformer.insert(0, "Model", "Transformer")
transformer["Source"] = "transformer.ipynb (stored test metrics_summary)"
transformer = pd.concat([transformer_1, transformer], ignore_index=True)

print("CNN-LSTM")
display(cnn)
print("Transformer")
display(transformer)


CNN-LSTM


,Model,Horizon_Days,MAE,RMSE,R2,MAPE,MSE,MBE,Source
0,CNN-LSTM,1,3.166,4.880,0.475,42.110,23.813,0.124,cnn_lstm.ipynb (stored Horizon 1d print)
1,CNN-LSTM,7,3.827,6.393,0.106,45.023,40.866,-1.063,cnn_lstm.ipynb (stored test metrics_summary)
2,CNN-LSTM,14,3.912,6.648,0.028,45.759,44.198,-1.068,cnn_lstm.ipynb (stored test metrics_summary)
3,CNN-LSTM,30,4.160,6.638,-0.030,55.659,44.061,-0.168,cnn_lstm.ipynb (stored test metrics_summary)


Transformer


,Model,Horizon_Days,MAE,RMSE,R2,MAPE,MSE,MBE,Source
0,Transformer,1,4.364,6.773,-0.011,56.940,45.875,-0.178,transformer.ipynb (stored Horizon 1d (test) pr...
1,Transformer,7,4.563,6.485,0.080,64.775,42.057,0.684,transformer.ipynb (stored test metrics_summary)
2,Transformer,14,4.231,6.701,0.012,54.947,44.903,-0.254,transformer.ipynb (stored test metrics_summary)
3,Transformer,30,4.436,6.353,0.056,65.957,40.358,0.989,transformer.ipynb (stored test metrics_summary)


## Random Forest and XGBoost pooled-path CSVs

Pooled-path CSVs for 7 / 14 / 30, plus the 1-day specialist scored on `*_predictions_1d.csv`.


In [18]:
rf = pd.read_csv(RF_CSV)
xgb = pd.read_csv(XGB_CSV)

tree_cols = [
    "Horizon_Days",
    "Model",
    "MAE",
    "RMSE",
    "R2",
    "MAPE",
    "MSE",
    "MBE",
]


def lead1_from_pred_csv(pred_csv, model_name, source):
    df = pd.read_csv(pred_csv)
    y_true = df["Actual_PM25"].to_numpy(dtype=float)
    y_pred = df["Predicted_PM25"].to_numpy(dtype=float)
    mse = mean_squared_error(y_true, y_pred)
    return pd.DataFrame(
        [
            {
                "Horizon_Days": 1,
                "Model": model_name,
                "MAE": mean_absolute_error(y_true, y_pred),
                "MSE": mse,
                "RMSE": float(np.sqrt(mse)),
                "R2": float(r2_score(y_true, y_pred)),
                "MAPE": float(
                    np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))
                    * 100
                ),
                "MBE": float(np.mean(y_pred - y_true)),
                "Source": source,
            }
        ]
    )


rf_cmp = rf[tree_cols].copy()
rf_cmp["Source"] = str(RF_CSV)
rf_1 = lead1_from_pred_csv(
    RF_PRED / "rf_predictions_1d.csv",
    "Random Forest",
    str(RF_PRED / "rf_predictions_1d.csv"),
)
rf_cmp = pd.concat([rf_1, rf_cmp], ignore_index=True)

xgb_cmp = xgb[tree_cols].copy()
xgb_cmp["Source"] = str(XGB_CSV)
xgb_1 = lead1_from_pred_csv(
    XGB_PRED / "xgb_predictions_1d.csv",
    "XGBoost",
    str(XGB_PRED / "xgb_predictions_1d.csv"),
)
xgb_cmp = pd.concat([xgb_1, xgb_cmp], ignore_index=True)

display(rf_cmp)
display(xgb_cmp)


,Horizon_Days,Model,MAE,MSE,RMSE,R2,MAPE,MBE,Source
0,1,Random Forest,3.153571,26.030224,5.101982,0.493929,41.353007,0.064154,rf_results_path/rf_predictions_1d.csv
1,7,Random Forest,4.061854,39.314798,6.270151,0.243042,51.357212,-0.238135,rf_results_path/random_forest_pooled_path_resu...
2,14,Random Forest,4.204784,42.269277,6.501483,0.178567,52.374212,-0.366805,rf_results_path/random_forest_pooled_path_resu...
3,30,Random Forest,4.333896,45.050183,6.711943,0.092781,54.561768,-0.338272,rf_results_path/random_forest_pooled_path_resu...


,Horizon_Days,Model,MAE,MSE,RMSE,R2,MAPE,MBE,Source
0,1,XGBoost,3.155274,26.761653,5.173167,0.479709,40.898101,-0.021990,xgb_results_path/xgb_predictions_1d.csv
1,7,XGBoost,4.029561,40.280787,6.346715,0.224443,49.835307,-0.419409,xgb_results_path/xgboost_pooled_path_results.csv
2,14,XGBoost,4.192466,43.778884,6.616561,0.149230,50.923313,-0.585515,xgb_results_path/xgboost_pooled_path_results.csv
3,30,XGBoost,4.275382,46.152142,6.793537,0.070590,52.273338,-0.606966,xgb_results_path/xgboost_pooled_path_results.csv


## LSTM and GRU pooled-path CSV

From `gru_lstm_multi_horizon.ipynb`. One multi-output net per H, 30-day input, original µg/m³. Horizons 7 / 14 / 30 only. No 1-day run.


In [19]:
mh = pd.read_csv(LSTM_GRU_CSV).rename(
    columns={"Horizon": "Horizon_Days", "MAPE (%)": "MAPE", "R²": "R2"}
)
mh_cols = ["Horizon_Days", "Model", "MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"]
lstm_cmp = mh.loc[mh["Model"] == "LSTM", mh_cols].copy()
lstm_cmp["Source"] = str(LSTM_GRU_CSV)
gru_cmp = mh.loc[mh["Model"] == "GRU", mh_cols].copy()
gru_cmp["Source"] = str(LSTM_GRU_CSV)

display(lstm_cmp)
display(gru_cmp)


,Horizon_Days,Model,MAE,RMSE,R2,MAPE,MSE,MBE,Source
0,7,LSTM,4.321137,7.084732,0.031968,47.442724,50.193433,-1.391339,true_multi_horizon_results/multi_horizon_metri...
1,14,LSTM,4.415254,7.386747,-0.055707,47.050342,54.564024,-1.664981,true_multi_horizon_results/multi_horizon_metri...
2,30,LSTM,4.281925,7.185571,-0.035566,47.415896,51.632432,-1.446341,true_multi_horizon_results/multi_horizon_metri...


,Horizon_Days,Model,MAE,RMSE,R2,MAPE,MSE,MBE,Source
3,7,GRU,4.136066,6.520930,0.179909,48.066383,42.522523,-0.969420,true_multi_horizon_results/multi_horizon_metri...
4,14,GRU,4.049140,6.819694,0.100157,42.158923,46.508224,-1.745710,true_multi_horizon_results/multi_horizon_metri...
5,30,GRU,4.495202,7.179877,-0.033925,51.399740,51.550633,-1.444278,true_multi_horizon_results/multi_horizon_metri...


## Combined comparison table


In [20]:
comparison_df = pd.concat(
    [cnn, transformer, lstm_cmp, gru_cmp, rf_cmp, xgb_cmp],
    ignore_index=True,
)
comparison_df["Evaluation"] = np.where(
    comparison_df["Horizon_Days"] == 1, "lead_1", "pooled_path"
)
comparison_df["Units"] = "ug/m3 (MAPE in %)"

comparison_df = comparison_df[
    [
        "Model",
        "Horizon_Days",
        "Evaluation",
        "MAE",
        "RMSE",
        "MSE",
        "R2",
        "MAPE",
        "MBE",
        "Source",
        "Units",
    ]
].sort_values(["Horizon_Days", "MAE"]).reset_index(drop=True)

display(comparison_df.round(4))

out_path = HERE / "pooled_path_comparison.csv"
comparison_df.to_csv(out_path, index=False)
print("Saved", out_path.resolve())


,Model,Horizon_Days,Evaluation,MAE,RMSE,MSE,R2,MAPE,MBE,Source,Units
0,Random Forest,1,lead_1,3.1536,5.1020,26.0302,0.4939,41.3530,0.0642,rf_results_path/rf_predictions_1d.csv,ug/m3 (MAPE in %)
1,XGBoost,1,lead_1,3.1553,5.1732,26.7617,0.4797,40.8981,-0.0220,xgb_results_path/xgb_predictions_1d.csv,ug/m3 (MAPE in %)
2,CNN-LSTM,1,lead_1,3.1660,4.8800,23.8130,0.4750,42.1100,0.1240,cnn_lstm.ipynb (stored Horizon 1d print),ug/m3 (MAPE in %)
3,Transformer,1,lead_1,4.3640,6.7730,45.8750,-0.0110,56.9400,-0.1780,transformer.ipynb (stored Horizon 1d (test) pr...,ug/m3 (MAPE in %)
4,CNN-LSTM,7,pooled_path,3.8270,6.3930,40.8660,0.1060,45.0230,-1.0630,cnn_lstm.ipynb (stored test metrics_summary),ug/m3 (MAPE in %)
5,XGBoost,7,pooled_path,4.0296,6.3467,40.2808,0.2244,49.8353,-0.4194,xgb_results_path/xgboost_pooled_path_results.csv,ug/m3 (MAPE in %)
6,Random Forest,7,pooled_path,4.0619,6.2702,39.3148,0.2430,51.3572,-0.2381,rf_results_path/random_forest_pooled_path_resu...,ug/m3 (MAPE in %)
7,GRU,7,pooled_path,4.1361,6.5209,42.5225,0.1799,48.0664,-0.9694,true_multi_horizon_results/multi_horizon_metri...,ug/m3 (MAPE in %)
8,LSTM,7,pooled_path,4.3211,7.0847,50.1934,0.0320,47.4427,-1.3913,true_multi_horizon_results/multi_horizon_metri...,ug/m3 (MAPE in %)
9,Transformer,7,pooled_path,4.5630,6.4850,42.0570,0.0800,64.7750,0.6840,transformer.ipynb (stored test metrics_summary),ug/m3 (MAPE in %)


Saved /Users/matthewbutler/Documents/MastersPaper/code/model_training/pooled_path_comparison.csv


## Checks

1. CNN-LSTM / transformer rows match the printed Horizon 1d / 7d / 14d / 30d test blocks in those notebooks.
2. RF / XGBoost rows match a fresh score over `*_predictions_{h}d.csv` (H=1 is that file alone; 7 / 14 / 30 are pooled).
3. LSTM / GRU rows match `multi_horizon_metrics_original_scale.csv` and a fresh score over `{lstm,gru}_{H}day_forecast_horizon.csv`.


In [21]:
def printed_test_metrics(nb_path):
    nb = json.loads(Path(nb_path).read_text())
    text = []
    for cell in nb["cells"]:
        for out in cell.get("outputs", []):
            if out.get("output_type") != "stream":
                continue
            text.append("".join(out.get("text", [])))
    blob = "\n".join(text)
    rows = {}
    for h in (1, 7, 14, 30):
        key = None
        if h == 1:
            candidates = ("Horizon 1d (test)", "Horizon 1d\n")
        else:
            candidates = (f"Horizon {h}d (test)", f"Horizon {h}d")
        for candidate in candidates:
            if candidate in blob:
                key = candidate
                break
        if key is None:
            raise ValueError(f"Missing Horizon {h}d in {nb_path}")
        chunk = blob.split(key, 1)[1]
        got = {}
        for line in chunk.splitlines()[:8]:
            line = line.strip()
            if line.startswith("MAE:"):
                got["MAE"] = float(line.split()[1])
            elif line.startswith("RMSE:"):
                got["RMSE"] = float(line.split()[1])
            elif line.startswith("R2:") or line.startswith("R"):
                got["R2"] = float(line.split()[1])
            elif line.startswith("MAPE:"):
                got["MAPE"] = float(line.split()[1].replace("%", ""))
            elif line.startswith("MSE:"):
                got["MSE"] = float(line.split()[1])
            elif line.startswith("MBE:"):
                got["MBE"] = float(line.split()[1])
        rows[h] = got
    return rows


def path_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(
            np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
        ),
        "MBE": float(np.mean(y_pred - y_true)),
    }


def recompute_pooled(pred_dir, prefix, H):
    dfs = {
        h: pd.read_csv(pred_dir / f"{prefix}_{h}d.csv", parse_dates=["Forecast_Origin"])
        for h in range(1, H + 1)
    }
    common = set.intersection(*[set(dfs[h]["Forecast_Origin"]) for h in range(1, H + 1)])
    y_true, y_pred = [], []
    for origin in sorted(common):
        for h in range(1, H + 1):
            sub = dfs[h].loc[dfs[h]["Forecast_Origin"] == origin]
            y_true.append(sub["Actual_PM25"].iloc[0])
            y_pred.append(sub["Predicted_PM25"].iloc[0])
    return path_metrics(y_true, y_pred)


checks = []

for name, nb_path, subset in [
    ("CNN-LSTM", CNN_NB, cnn),
    ("Transformer", TRANS_NB, transformer),
]:
    printed = printed_test_metrics(nb_path)
    for _, row in subset.iterrows():
        h = int(row["Horizon_Days"])
        for col in ["MAE", "RMSE", "R2", "MSE", "MBE"]:
            ok = abs(row[col] - printed[h][col]) < 0.0015
            checks.append((ok, f"{name} H={h} {col} table={row[col]} print={printed[h][col]}"))
        ok = abs(row["MAPE"] - printed[h]["MAPE"]) < 0.015
        checks.append((ok, f"{name} H={h} MAPE table={row['MAPE']} print={printed[h]['MAPE']}"))

for label, csv_df, pred_dir, prefix in [
    ("Random Forest", rf_cmp, RF_PRED, "rf_predictions"),
    ("XGBoost", xgb_cmp, XGB_PRED, "xgb_predictions"),
]:
    for H in (1, 7, 14, 30):
        recomputed = recompute_pooled(pred_dir, prefix, H)
        row = csv_df.loc[csv_df["Horizon_Days"] == H].iloc[0]
        for col in ["MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"]:
            ok = abs(float(row[col]) - recomputed[col]) < 1e-6
            checks.append(
                (ok, f"{label} H={H} {col} csv={row[col]} recomputed={recomputed[col]}")
            )

for name, cmp_df, prefix in [("LSTM", lstm_cmp, "lstm"), ("GRU", gru_cmp, "gru")]:
    for _, row in cmp_df.iterrows():
        h = int(row["Horizon_Days"])
        src = mh.loc[(mh["Model"] == name) & (mh["Horizon_Days"] == h)].iloc[0]
        for col in ["MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"]:
            ok = abs(float(row[col]) - float(src[col])) < 1e-12
            checks.append((ok, f"{name} H={h} {col} table vs metrics csv"))
        pred = pd.read_csv(LSTM_GRU_PRED / f"{prefix}_{h}day_forecast_horizon.csv")
        recomputed = path_metrics(pred["Actual_PM25"], pred["Predicted_PM25"])
        for col in ["MAE", "RMSE", "R2", "MSE", "MBE"]:
            ok = abs(float(row[col]) - recomputed[col]) < 1e-6
            checks.append(
                (ok, f"{name} H={h} {col} csv={row[col]} recomputed={recomputed[col]}")
            )

failed = [msg for ok, msg in checks if not ok]
print(f"{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("\n".join(failed))
print("All checks passed.")


162/162 checks passed
All checks passed.
